In [1]:
import numpy as np
from collections import Counter

print("========== CUSTOM BYTE PAIR ENCODING TOKENIZER ==========\n")

# -----------------------------------------------------
# Step 1 : Create Corpus
# -----------------------------------------------------

corpus = [
    "study",
    "student",
    "students",
    "studying",
    "teach",
    "teacher",
    "teaching",
    "taught",
    "learn",
    "learner",
    "learning",
    "learned"
]

print("Corpus:\n")
print(corpus)

# -----------------------------------------------------
# Step 2 : Build Vocabulary
# -----------------------------------------------------

vocab = {}

for word in corpus:
    chars = tuple(list(word) + ["</w>"])
    vocab[chars] = vocab.get(chars, 0) + 1

print("\nInitial Vocabulary:\n")

for word, freq in vocab.items():
    print(word, ":", freq)

# -----------------------------------------------------
# Step 3 : Count Pair Frequencies
# -----------------------------------------------------

def get_pair_frequencies(vocab):

    pairs = Counter()

    for word, freq in vocab.items():

        symbols = list(word)

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq

    return pairs

# -----------------------------------------------------
# Step 4 : Merge Function
# -----------------------------------------------------

def merge_vocab(pair, vocab):

    new_vocab = {}

    for word, freq in vocab.items():

        merged = []
        i = 0

        while i < len(word):

            if (
                i < len(word)-1
                and word[i] == pair[0]
                and word[i+1] == pair[1]
            ):
                merged.append(word[i] + word[i+1])
                i += 2
            else:
                merged.append(word[i])
                i += 1

        new_vocab[tuple(merged)] = freq

    return new_vocab

# -----------------------------------------------------
# Step 5 : Find First Best Pair
# -----------------------------------------------------

pairs = get_pair_frequencies(vocab)

best_pair = max(pairs, key=pairs.get)

print("\nMost Frequent Pair :", best_pair)
print("Frequency :", pairs[best_pair])

# -----------------------------------------------------
# Step 6 : Merge Once
# -----------------------------------------------------

vocab = merge_vocab(best_pair, vocab)

print("\nVocabulary After First Merge:\n")

for word, freq in vocab.items():
    print(word, ":", freq)

# -----------------------------------------------------
# Step 7 : Learn Merge Rules
# -----------------------------------------------------

num_merges = 10

merge_rules = []

print("\nLearning Merge Rules...\n")

for i in range(num_merges):

    pairs = get_pair_frequencies(vocab)

    if not pairs:
        break

    best_pair = max(pairs, key=pairs.get)

    merge_rules.append(best_pair)

    print(f"Merge {i+1}: {best_pair}  Frequency = {pairs[best_pair]}")

    vocab = merge_vocab(best_pair, vocab)

# -----------------------------------------------------
# Step 8 : Display Merge Rules
# -----------------------------------------------------

print("\nLearned Merge Rules:\n")

for i, rule in enumerate(merge_rules, start=1):
    print(f"{i}. {rule}")

# -----------------------------------------------------
# Step 9 : Final Vocabulary
# -----------------------------------------------------

print("\nFinal Vocabulary:\n")

for word, freq in vocab.items():
    print(word, ":", freq)

# -----------------------------------------------------
# Step 10 : Build Token Vocabulary
# -----------------------------------------------------

final_vocab = set()

for word in vocab:

    for token in word:
        final_vocab.add(token)

print("\nVocabulary Tokens:\n")

for token in sorted(final_vocab):
    print(token)

print("\nVocabulary Size :", len(final_vocab))

# -----------------------------------------------------
# Step 11 : Encode Function
# -----------------------------------------------------

def encode(word, merge_rules):

    tokens = list(word.lower()) + ["</w>"]

    for pair in merge_rules:

        new_tokens = []
        i = 0

        while i < len(tokens):

            if (
                i < len(tokens)-1
                and tokens[i] == pair[0]
                and tokens[i+1] == pair[1]
            ):
                new_tokens.append(pair[0] + pair[1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return tokens

# -----------------------------------------------------
# Step 12 : Decode Function
# -----------------------------------------------------

def decode(tokens):

    word = ""

    for token in tokens:

        if token != "</w>":
            word += token.replace("</w>", "")

    return word

# -----------------------------------------------------
# Step 13 : Test Encode & Decode
# -----------------------------------------------------

test_words = [
    "student",
    "students",
    "studying",
    "teacher",
    "teaching",
    "learner",
    "learning"
]

print("\n========== TEST RESULTS ==========\n")

for word in test_words:

    encoded = encode(word, merge_rules)

    decoded = decode(encoded)

    print("-" * 40)
    print("Original :", word)
    print("Encoded  :", encoded)
    print("Decoded  :", decoded)

print("\n========== PROGRAM COMPLETED ==========")

========== CUSTOM BYTE PAIR ENCODING TOKENIZER ==========

Corpus:

['study', 'student', 'students', 'studying', 'teach', 'teacher', 'teaching', 'taught', 'learn', 'learner', 'learning', 'learned']

Initial Vocabulary:

('s', 't', 'u', 'd', 'y', '</w>') : 1
('s', 't', 'u', 'd', 'e', 'n', 't', '</w>') : 1
('s', 't', 'u', 'd', 'e', 'n', 't', 's', '</w>') : 1
('s', 't', 'u', 'd', 'y', 'i', 'n', 'g', '</w>') : 1
('t', 'e', 'a', 'c', 'h', '</w>') : 1
('t', 'e', 'a', 'c', 'h', 'e', 'r', '</w>') : 1
('t', 'e', 'a', 'c', 'h', 'i', 'n', 'g', '</w>') : 1
('t', 'a', 'u', 'g', 'h', 't', '</w>') : 1
('l', 'e', 'a', 'r', 'n', '</w>') : 1
('l', 'e', 'a', 'r', 'n', 'e', 'r', '</w>') : 1
('l', 'e', 'a', 'r', 'n', 'i', 'n', 'g', '</w>') : 1
('l', 'e', 'a', 'r', 'n', 'e', 'd', '</w>') : 1

Most Frequent Pair : ('e', 'a')
Frequency : 7

Vocabulary After First Merge:

('s', 't', 'u', 'd', 'y', '</w>') : 1
('s', 't', 'u', 'd', 'e', 'n', 't', '</w>') : 1
('s', 't', 'u', 'd', 'e', 'n', 't', 's', '</w>') : 1
(